# New version, using the .parquet exports
instead of synchronizing the complete postgresql database all the time, i can export PARAM and METRIC .parquet files per group_id and only synchronize/copy what i need.

In [1]:
import sys
import os
project_root = os.path.abspath('..')          # one level up from notebooks' directory
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import numpy as np
from pathlib import Path
from key_mapping import key_mapping
from data_preparation import add_grouped_mean_columns, get_multi_carrier_metrics

updating value_mapping...


In [2]:
# pandas version
def make_single_step_multi_carrier_plot_data(params_file:str):
    from key_mapping import key_mapping
    from data_preparation import add_grouped_mean_columns, get_multi_carrier_metrics
    print(f'Reading params file {params_file}')
    PARAMS = pd.read_parquet(f"../data/experiments/exports/{params_file}", )
    print('Reading metrics file')
    METRICS = pd.read_parquet(f"../data/experiments/exports/{params_file.replace('params', 'metrics')}", )

    print('Pivot METRICS')
    METRICS_WIDE = METRICS.pivot(columns='key', index=['run_uuid', 'step'], values='value').reset_index()
    print('Get single step metrics')
    single_step_metrics = [col for col in METRICS_WIDE.columns if pd.isna(METRICS_WIDE[METRICS_WIDE['step'] > 0][col]).all()]
    SS_METRICS = METRICS_WIDE[METRICS_WIDE['step'] == 0][['run_uuid'] + single_step_metrics]
    SS_MERGED = pd.merge(PARAMS, SS_METRICS, on='run_uuid')
    print('Add grouped columns')
    add_grouped_mean_columns(SS_MERGED, inplace=True)

    # single step, multi-carrier metrics
    print('Get multi-carrier metrcis')
    SS_MC_METRICS = get_multi_carrier_metrics(SS_METRICS)
    SS_MC_MERGED = pd.merge(PARAMS, SS_MC_METRICS, on='run_uuid')

    # Multi-step metrics
    # print('Get multi-step metrics')
    # multi_step_metrics = [col for col in METRICS_WIDE.columns if col not in single_step_metrics]
    # MS_METRICS = METRICS_WIDE[multi_step_metrics]
    # MS_METRICS.update(MS_METRICS.groupby('run_uuid').ffill())  # forward fill the multi-step metrics that might have gaps
    # MS_METRICS = MS_METRICS.replace(np.inf, np.nan)
    # add_grouped_mean_columns(MS_METRICS, inplace=True)
    # MS_MERGED = pd.merge(PARAMS, MS_METRICS, on='run_uuid')

    # to wide format
    print('Pivot SS_MC_MERGED to wide format')
    SS_MC_MERGED_WIDE:pd.DataFrame = SS_MC_MERGED.pivot(columns='key', values='value', index=PARAMS.columns.to_list() + ['carrier']).reset_index()
    # SS_MC_MERGED_WIDE = SS_MC_MERGED_WIDE.rename(columns=key_mapping)

    print('Write parquet file to disk')
    write_file =params_file.split('-')[0] + '-SS_MC_MERGED_WIDE.parquet'
    write_file = f"../data/experiments/plot_ready/{write_file}"
    SS_MC_MERGED_WIDE.to_parquet(write_file)
    print(f'Done. File is at {write_file}')

In [3]:
## Generating the plot-ready files
# for foo in ['14', '15', '16', '17', '19', '20', '21', '22', '23']:
#     params_file = f"['P3.1.{foo}']-params.parquet"
#     make_single_step_multi_carrier_plot_data(params_file)

## Plots

In [4]:
df = [
    pd.read_parquet(
        f"../data/experiments/plot_ready/['P3.1.{tag_postfix}']-SS_MC_MERGED_WIDE.parquet"
    )
    for tag_postfix in [
        "23",  # n=8
        "14", 
        "15", 
        "16",  # n=16 from here on
        "17", 
        "19", 
        "20", 
        "21", 
        "22"
        ]
]
df = pd.concat(df)


In [5]:
# ADDING SOME COLUMNS
# add the alpha column (num_unknown_orders as a fraction of the actual orders to estimate) 
df['auction__bundling_and_bidding__fitness_function__bundle_fitness__frac_unknown_orders'] = df['auction__bundling_and_bidding__fitness_function__bundle_fitness__num_unknown_orders'] / (df['data__num_requests_per_carrier'] * df['auction__num_submitted_requests'])

# adding normalized pompeiu hausdorff: since the max distance can be the diagonal in the 100 by 100 square, we divide by np.sqrt(100**2 + 100**2)
# df['$\hat{d}_{PHD}$'] = df['$d_{PHD}$']/np.sqrt(100**2 + 100**2)
df['norm_my_hausdorff_distance'] = df['my_hausdorff_distance']/np.sqrt(100**2 + 100**2)
# df['$\hat{d}_{MPHD}$'] = df['$d_{MPHD}$']/np.sqrt(100**2 + 100**2)
df['norm_my_modified_hausdorff_distance'] = df['my_modified_hausdorff_distance']/np.sqrt(100**2 + 100**2)

# REPLACE
df.replace({'# clusters': {'3': 'Clustered', 'None': 'Random'}}, inplace=True)

In [6]:
# plot_cols = list(
    # df.columns[
        # list(df.columns).index("mlflow.user") + 1 :
    # ]
# )
exclude_col = ['run_uuid',
 'name_x',
 'start_time',
 'end_time',
 'artifact_uri',
 'experiment_id',
 'name_y',
 'artifact_location',
 'creation_time',
 'last_update_time',
#  'group_id',
#  'instance_id',
 'mlflow.loggedArtifacts',
 'mlflow.runName',
 'mlflow.source.git.commit',]
plot_cols = [col for col in df.columns if col not in exclude_col]
plot_cols = [col for col in plot_cols if df[col].nunique() > 1]  # exclude constant-value columns
plot_cols = [None] + plot_cols

# TEMPORARY: use all cols
# plot_cols = df.columns

In [7]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotnine as p9
import os
from my_theming import my_p9_theme, my_scale_color_and_fill

# --- 1. Global holder for the current plot object ---
current_p = [None]


# --- 2. Define the plotting function logic ---
def create_plot(df, x, y, fill, linetype, cols, rows, hide_x_axis, scale_y_log10, ylim_min, ylim_max, _trigger):
    # --- NEW: Intercept and apply active dynamic filters ---
    filtered_df = df.copy()
    for col, widget in filter_widgets.items():
        val = widget.value
        if isinstance(widget, widgets.SelectMultiple):
            if val:  # If items are selected, filter by selection
                filtered_df = filtered_df[filtered_df[col].isin(val)]
        elif isinstance(widget, widgets.FloatRangeSlider):
            filtered_df = filtered_df[
                (filtered_df[col] >= val[0]) & (filtered_df[col] <= val[1])
            ]
            
    if filtered_df.empty:
        print("⚠️ No data matches the selected filters. Adjust your filters above.")
        return

    # (Setting up labels and aesthetics - same as your logic)
    ylab = filtered_df["Error Function"].iloc[0] if y == "target_opt_final" else y

    aes_kwargs = {"x": x, "y": y}
    if fill not in [None, "None"]:
        aes_kwargs["fill"] = fill
    if linetype not in [None, "None"]:
        aes_kwargs["linetype"] = linetype

    my_aes = p9.aes(**aes_kwargs)

    p = (
        p9.ggplot(filtered_df, my_aes)
        + p9.geom_boxplot()
        + my_p9_theme
        + p9.theme(figure_size=(12, 6))
        + my_scale_color_and_fill
        + p9.labs(y=ylab)
    )

    if scale_y_log10:
        p += p9.scale_y_log10()
    if hide_x_axis:
        p += p9.theme(
            axis_text_x=p9.element_blank(),
            axis_ticks_x=p9.element_blank(),
            axis_title_x=p9.element_blank(),
            panel_grid_major_x=p9.element_blank(),
            panel_grid_minor_x=p9.element_blank(),
        )
    if ylim_min != ylim_max:
        p += p9.ylim(ylim_min, ylim_max)

    if cols or rows:
        p += p9.facet_grid(rows=rows, cols=cols)

    # Update the global reference for the save button
    current_p[0] = p

    # --- CRITICAL CHANGE START ---
    # In interactive_output, we must explicitly display the plot
    display(p)

    # Print stats 
    group_keys = [
        key for key in [fill, cols, rows, linetype] if key not in [None, "None"]
    ]
    if group_keys:
        n_per_series = filtered_df.groupby(group_keys)[y].describe()
        pd.set_option('display.max_columns', len(df.columns))
        pd.set_option('display.width', 1000) # Setting a very large width
        print(f"\nn per boxplot:\n{n_per_series}")
    # --- CRITICAL CHANGE END ---


# --- 3. Define Widgets (Keep these exactly as they were) ---
ui_x = widgets.Dropdown(
    options=plot_cols, value=None, description="X Axis:"
)
ui_y = widgets.Dropdown(
    options=plot_cols, value="my_hausdorff_distance", description="Y Axis:"
)
ui_fill = widgets.Dropdown(
    options=plot_cols, value="Optimizer", description="Fill Color:"
)
ui_line = widgets.Dropdown(
    options=plot_cols, value="Optimizer", description="Linetype:"
)
ui_cols = widgets.Dropdown(
    options= plot_cols, value="# clusters", description="Facet Cols:"
)
ui_rows = widgets.Dropdown(
    options=plot_cols,
    # value="data__num_requests_per_carrier",
    description="Facet Rows:",
)
ui_hide_x = widgets.Checkbox(value=False, description="Hide x axis ticks")
ui_log = widgets.Checkbox(value=False, description="Log y axis")
ui_ylim_min = widgets.FloatText(value=0, description='ylim min:', disabled=False)
ui_ylim_max = widgets.FloatText(value=None, description='ylim max:', disabled=False)

ui_filename = widgets.Text(value="my_plot.png", description="Filename:")
ui_save_btn = widgets.Button(description="💾 Save Figure", button_style="success")
ui_save_output = widgets.Output()

# --- NEW: Dynamic Filtering State and Controls ---
filter_widgets = {}  # Tracks col_name -> widget mapping
filters_box = widgets.VBox([])  # Container to hold rows of active filter widgets

ui_filter_col = widgets.Dropdown(options=plot_cols, description="Add filter:")
ui_add_filter_btn = widgets.Button(description="➕ Add Filter", button_style="info")
# Hidden trigger widget to force interactive_output to rerun
ui_filter_trigger = widgets.IntText(value=0, layout=widgets.Layout(display='none'))

def add_filter_clicked(b):
    col = ui_filter_col.value
    if not col or col in filter_widgets:
        return  # Prevent adding duplicate filters for the same column
    
    # Check data type to decide widget style
    sample_data = df[col]
    # a) treat all as categorical
    unique_vals = sorted(list(sample_data.dropna().unique()))
    w = widgets.SelectMultiple(
            options=unique_vals, value=unique_vals, description=col
        )
    
    # b) sliders for numerical, selectors for categorical
    # if sample_data.dtype in ['object', 'category', 'bool']:
    #     unique_vals = sorted(list(sample_data.dropna().unique()))
    #     w = widgets.SelectMultiple(
    #         options=unique_vals, value=unique_vals, description=col
    #     )
    # else:  # Numerical columns
    #     c_min, c_max = float(sample_data.min()), float(sample_data.max())
    #     w = widgets.FloatRangeSlider(
    #         min=c_min, max=c_max, value=(c_min, c_max), description=col
    #     )
    
    # Watch for changes on the dynamic widget to increment our main trigger
    def trigger_update(change):
        if change['name'] == 'value':
            ui_filter_trigger.value += 1
    w.observe(trigger_update)
    
    # Create a small button to delete this specific filter line
    btn_remove = widgets.Button(description="❌", layout=widgets.Layout(width='40px'))
    row = widgets.HBox([w, btn_remove])
    
    def remove_clicked(rb):
        if col in filter_widgets:
            del filter_widgets[col]
        filters_box.children = [r for r in filters_box.children if r != row]
        ui_filter_trigger.value += 1  # Refresh plot after deletion
        
    btn_remove.on_click(remove_clicked)
    
    # Save widget reference and add to view hierarchy
    filter_widgets[col] = w
    filters_box.children = list(filters_box.children) + [row]
    ui_filter_trigger.value += 1  # Trigger initial draw with new filter active

ui_add_filter_btn.on_click(add_filter_clicked)


def on_save_clicked(b):
    with ui_save_output:
        clear_output()
        if current_p[0] is not None:
            current_p[0].save(ui_filename.value, verbose=False)
            print(f"✅ Saved!")
        else:
            print("❌ Error")


ui_save_btn.on_click(on_save_clicked)

# --- 4. Linking and Displaying ---
out = widgets.interactive_output(
    create_plot,
    {
        "df": widgets.fixed(df),
        "x": ui_x,
        "y": ui_y,
        "fill": ui_fill,
        "linetype": ui_line,
        "cols": ui_cols,
        "rows": ui_rows,
        "hide_x_axis": ui_hide_x,
        "scale_y_log10": ui_log,
        "ylim_min": ui_ylim_min,
        "ylim_max": ui_ylim_max,
        "_trigger": ui_filter_trigger,
    },
)

# Bundle our dynamic filter builder items into its own sub-layout block
filter_ui_panel = widgets.VBox([
    widgets.HBox([ui_filter_col, ui_add_filter_btn]),
    filters_box
])

controls = widgets.VBox(
    [
        filter_ui_panel,
        widgets.HBox([ui_x, ui_y, ui_fill]),
        widgets.HBox([ui_line, ui_cols, ui_rows]),
        widgets.HBox([ui_hide_x, ui_log]),
        widgets.HBox([ui_ylim_min, ui_ylim_max]),
        widgets.HBox([ui_filename, ui_save_btn, ui_save_output]),
    ]
)

display(controls, out)

Output()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotnine as p9
import os
from my_theming import my_p9_theme, my_scale_color_and_fill

# --- 1. Global holder for the current plot object ---
current_p = [None]


# --- 2. Define the plotting function logic ---
def create_plot(df, x, y, fill, linetype, cols, rows, hide_x_axis, scale_y_log10,
                 ylim_min, ylim_max, fig_width, fig_height, legend_pos, _trigger):
    # --- NEW: Intercept and apply active dynamic filters ---
    filtered_df = df.copy()
    for col, widget in filter_widgets.items():
        val = widget.value
        if isinstance(widget, widgets.SelectMultiple):
            if val:  # If items are selected, filter by selection
                filtered_df = filtered_df[filtered_df[col].isin(val)]
        elif isinstance(widget, widgets.FloatRangeSlider):
            filtered_df = filtered_df[
                (filtered_df[col] >= val[0]) & (filtered_df[col] <= val[1])
            ]
            
    if filtered_df.empty:
        print("⚠️ No data matches the selected filters. Adjust your filters above.")
        return

    # (Setting up labels and aesthetics - same as your logic)
    ylab = filtered_df["Error Function"].iloc[0] if y == "target_opt_final" else y

    aes_kwargs = {"x": x, "y": y}
    if fill not in [None, "None"]:
        aes_kwargs["fill"] = fill
    if linetype not in [None, "None"]:
        aes_kwargs["linetype"] = linetype

    my_aes = p9.aes(**aes_kwargs)

    # --- NEW: legend_pos of "none" hides the legend entirely ---
    legend_position = False if legend_pos == "none" else legend_pos

    p = (
        p9.ggplot(filtered_df, my_aes)
        + p9.geom_boxplot()
        + my_p9_theme
        + p9.theme(figure_size=(fig_width, fig_height), legend_position=legend_position)
        + my_scale_color_and_fill
        + p9.labs(y=ylab)
    )

    if scale_y_log10:
        p += p9.scale_y_log10()
    if hide_x_axis:
        p += p9.theme(
            axis_text_x=p9.element_blank(),
            axis_ticks_x=p9.element_blank(),
            axis_title_x=p9.element_blank(),
            panel_grid_major_x=p9.element_blank(),
            panel_grid_minor_x=p9.element_blank(),
        )
    if ylim_min != ylim_max:
        p += p9.ylim(ylim_min, ylim_max)

    if cols or rows:
        p += p9.facet_grid(rows=rows, cols=cols)

    # Update the global reference for the save button
    current_p[0] = p

    # --- CRITICAL CHANGE START ---
    # In interactive_output, we must explicitly display the plot
    display(p)

    # Print stats 
    group_keys = [
        key for key in [fill, cols, rows, linetype] if key not in [None, "None"]
    ]
    if group_keys:
        n_per_series = filtered_df.groupby(group_keys)[y].describe()
        pd.set_option('display.max_columns', len(df.columns))
        pd.set_option('display.width', 1000) # Setting a very large width
        print(f"\nn per boxplot:\n{n_per_series}")
    # --- CRITICAL CHANGE END ---


# --- 3. Define Widgets (Keep these exactly as they were) ---
ui_x = widgets.Dropdown(
    options=plot_cols, value=None, description="X Axis:"
)
ui_y = widgets.Dropdown(
    options=plot_cols, value="my_hausdorff_distance", description="Y Axis:"
)
ui_fill = widgets.Dropdown(
    options=plot_cols, value="Optimizer", description="Fill Color:"
)
ui_line = widgets.Dropdown(
    options=plot_cols, value="Optimizer", description="Linetype:"
)
ui_cols = widgets.Dropdown(
    options= plot_cols, value="# clusters", description="Facet Cols:"
)
ui_rows = widgets.Dropdown(
    options=plot_cols,
    # value="data__num_requests_per_carrier",
    description="Facet Rows:",
)
ui_hide_x = widgets.Checkbox(value=False, description="Hide x axis ticks")
ui_log = widgets.Checkbox(value=False, description="Log y axis")
ui_ylim_min = widgets.FloatText(value=0, description='ylim min:', disabled=False)
ui_ylim_max = widgets.FloatText(value=None, description='ylim max:', disabled=False)

# --- NEW: figure size controls ---
ui_fig_width = widgets.FloatText(value=12, description='Fig width:', disabled=False)
ui_fig_height = widgets.FloatText(value=6, description='Fig height:', disabled=False)

# --- NEW: legend position control ---
ui_legend_pos = widgets.Dropdown(
    options=["right", "left", "top", "bottom", "none"],
    value="right",
    description="Legend pos:",
)

ui_filename = widgets.Text(value="my_plot.png", description="Filename:")
ui_save_btn = widgets.Button(description="💾 Save Figure", button_style="success")
ui_save_output = widgets.Output()

# --- NEW: Dynamic Filtering State and Controls ---
filter_widgets = {}  # Tracks col_name -> widget mapping
filters_box = widgets.VBox([])  # Container to hold rows of active filter widgets

ui_filter_col = widgets.Dropdown(options=plot_cols, description="Add filter:")
ui_add_filter_btn = widgets.Button(description="➕ Add Filter", button_style="info")
# Hidden trigger widget to force interactive_output to rerun
ui_filter_trigger = widgets.IntText(value=0, layout=widgets.Layout(display='none'))

def add_filter_clicked(b):
    col = ui_filter_col.value
    if not col or col in filter_widgets:
        return  # Prevent adding duplicate filters for the same column
    
    # Check data type to decide widget style
    sample_data = df[col]
    # a) treat all as categorical
    unique_vals = sorted(list(sample_data.dropna().unique()))
    w = widgets.SelectMultiple(
            options=unique_vals, value=unique_vals, description=col
        )
    
    # b) sliders for numerical, selectors for categorical
    # if sample_data.dtype in ['object', 'category', 'bool']:
    #     unique_vals = sorted(list(sample_data.dropna().unique()))
    #     w = widgets.SelectMultiple(
    #         options=unique_vals, value=unique_vals, description=col
    #     )
    # else:  # Numerical columns
    #     c_min, c_max = float(sample_data.min()), float(sample_data.max())
    #     w = widgets.FloatRangeSlider(
    #         min=c_min, max=c_max, value=(c_min, c_max), description=col
    #     )
    
    # Watch for changes on the dynamic widget to increment our main trigger
    def trigger_update(change):
        if change['name'] == 'value':
            ui_filter_trigger.value += 1
    w.observe(trigger_update)
    
    # Create a small button to delete this specific filter line
    btn_remove = widgets.Button(description="❌", layout=widgets.Layout(width='40px'))
    row = widgets.HBox([w, btn_remove])
    
    def remove_clicked(rb):
        if col in filter_widgets:
            del filter_widgets[col]
        filters_box.children = [r for r in filters_box.children if r != row]
        ui_filter_trigger.value += 1  # Refresh plot after deletion
        
    btn_remove.on_click(remove_clicked)
    
    # Save widget reference and add to view hierarchy
    filter_widgets[col] = w
    filters_box.children = list(filters_box.children) + [row]
    ui_filter_trigger.value += 1  # Trigger initial draw with new filter active

ui_add_filter_btn.on_click(add_filter_clicked)


def on_save_clicked(b):
    with ui_save_output:
        clear_output()
        if current_p[0] is not None:
            current_p[0].save(ui_filename.value, verbose=False)
            print(f"✅ Saved!")
        else:
            print("❌ Error")


ui_save_btn.on_click(on_save_clicked)

# --- 4. Linking and Displaying ---
out = widgets.interactive_output(
    create_plot,
    {
        "df": widgets.fixed(df),
        "x": ui_x,
        "y": ui_y,
        "fill": ui_fill,
        "linetype": ui_line,
        "cols": ui_cols,
        "rows": ui_rows,
        "hide_x_axis": ui_hide_x,
        "scale_y_log10": ui_log,
        "ylim_min": ui_ylim_min,
        "ylim_max": ui_ylim_max,
        "fig_width": ui_fig_width,
        "fig_height": ui_fig_height,
        "legend_pos": ui_legend_pos,
        "_trigger": ui_filter_trigger,
    },
)

# Bundle our dynamic filter builder items into its own sub-layout block
filter_ui_panel = widgets.VBox([
    widgets.HBox([ui_filter_col, ui_add_filter_btn]),
    filters_box
])

controls = widgets.VBox(
    [
        filter_ui_panel,
        widgets.HBox([ui_x, ui_y, ui_fill]),
        widgets.HBox([ui_line, ui_cols, ui_rows]),
        widgets.HBox([ui_hide_x, ui_log]),
        widgets.HBox([ui_ylim_min, ui_ylim_max]),
        widgets.HBox([ui_fig_width, ui_fig_height, ui_legend_pos]),
        widgets.HBox([ui_filename, ui_save_btn, ui_save_output]),
    ]
)

display(controls, out)

Output()

In [9]:
df.groupby(['group_id', 'auction__bundling_and_bidding__fitness_function__bundle_fitness__frac_unknown_orders', ])['my_hausdorff_distance'].describe()

,,count,mean,std,min,25%,50%,75%,max
group_id,auction__bundling_and_bidding__fitness_function__bundle_fitness__frac_unknown_orders,,,,,,,,
P3.1.14,1.0,19200.0,38.298419,15.545074,3.787640,27.133649,36.079663,46.942859,118.129793
P3.1.15,2.0,19200.0,49.392108,17.293884,9.566662,36.687633,47.028397,60.453380,127.922318
P3.1.16,0.5,19200.0,42.227751,14.340666,6.207520,31.741372,40.382561,51.285933,103.793133
P3.1.17,1.0,19200.0,44.930717,14.418323,9.158300,34.556139,42.426296,52.956981,120.666425
P3.1.19,2.0,4800.0,49.095858,15.508637,10.912437,37.224526,46.652373,59.295281,120.666425
P3.1.20,2.0,4800.0,48.318045,15.365031,11.974928,36.682756,45.839655,58.025414,116.540102
P3.1.21,2.0,4800.0,47.604796,15.101452,10.912437,36.261219,44.971107,56.682933,106.747841
P3.1.22,2.0,4800.0,47.605143,15.152821,10.912437,36.375563,45.135779,56.697441,107.884554
P3.1.23,0.5,19200.0,36.817431,18.952406,1.531733,22.922764,34.394639,47.987455,125.539859
